This notebook is used to add toxicity prediction to the top 20 recomended molecules

For antiviral and antibacterial drug development, these toxicity endpoints are particularly critical:

Essential Acute Toxicity Endpoints:
- Hepatotoxicity - Liver toxicity is one of the most common reasons drugs fail. The liver metabolizes most drugs, and anti-infectives often require high doses or prolonged treatment. Check for ALT/AST elevation predictions.
- Cardiotoxicity (hERG inhibition) - Blocking the hERG potassium channel can cause fatal cardiac arrhythmias. This is a major regulatory concern and frequent cause of drug withdrawal.
- Nephrotoxicity - Kidney toxicity is especially important since many drugs are renally cleared and infections can already stress kidney function.

Genotoxicity/Mutagenicity:
- AMES test prediction - Identifies potential DNA-damaging effects that could lead to cancer. Regulatory agencies require this data.
- Chromosome aberration - Another genotoxicity concern for long-term safety.
- Specific Considerations for Anti-infectives:
- Mitochondrial toxicity - Particularly important for antivirals since some (like older NRTIs) have caused mitochondrial dysfunction. Can lead to lactic acidosis, liver failure.
- Bone marrow suppression/Hematotoxicity - Many anti-infectives can suppress blood cell production, leading to anemia, neutropenia, or thrombocytopenia.
- Cytotoxicity against human cells - You want selectivity for pathogen over host cells. High general cytotoxicity is a red flag.

Drug-Drug Interaction Potential:
- CYP450 inhibition/induction - Patients with infections often take multiple medications. Strong CYP interactions can cause toxicity or reduce efficacy of other drugs.

In [1]:
import sys
import os

# Completely suppress stderr output
sys.stderr = open(os.devnull, 'w')

# Now import everything
import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import pandas as pd
import pubchempy as pcp

In [5]:
dataDir = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/'
modelBuildingDataDir = os.path.join(dataDir, 'modelBuildingData/')
resultsDir = os.path.join(dataDir, 'Results/')
saveDir = os.path.join(resultsDir, "AllVirus/")
os.makedirs(saveDir, exist_ok=True)

### Check toxicty for Enamine AntiviralsData set

In [6]:
EnamineAntiviralsData_predicted = pd.read_csv(saveDir + 'AllVirus_wART_EnamineDataset_predicted_all.csv')
max_val = EnamineAntiviralsData_predicted['pPotency_prediction'].max()
min_val = EnamineAntiviralsData_predicted['pPotency_prediction'].min()
print(f"pPotency_prediction range: {min_val:.3f} → {max_val:.3f}")

EnamineAntiviralsData_predicted

pPotency_prediction range: 4.557 → 6.352


,SMILES,pPotency_prediction,pPotency_std,pPotency_lower_95CI,pPotency_upper_95CI,IC50(M)_prediction,IC50(M)_lower_95CI,IC50(M)_upper_95CI
0,COCC(C)NC(=O)NC1=NN=C(S1)C2CC2,5.119991,0.674573,3.797828,6.442153,0.000008,3.612826e-07,0.000159
1,CN1C(=O)C=CN(CC(=O)NCC2CC2)C1=O,5.409013,0.674142,4.087694,6.730331,0.000004,1.860668e-07,0.000082
2,CCN(CC)CCNC(=O)C1=CC=CN=C1N2CCOCC2,4.906726,0.674204,3.585286,6.228166,0.000012,5.913362e-07,0.000260
3,CCC(CNC(=O)CC=1C(C)=NOC1C)N2CCCC2,4.879879,0.674231,3.558385,6.201372,0.000013,6.289675e-07,0.000276
4,CCC(C)(CNC(=O)CN1C=CC=CC1=O)N2CCOCC2,5.198782,0.674140,3.877469,6.520096,0.000006,3.019287e-07,0.000133
...,...,...,...,...,...,...,...,...
3195,CCOC=1N=CC=CC1NC(=O)NC2CCN3CCCCC23,5.240053,0.674091,3.918835,6.561272,0.000006,2.746173e-07,0.000121
3196,CC1=NN(CC(O)C=2C=CC=CC2)C(=O)C(C#N)=C1C,5.835830,0.674224,4.514350,7.157309,0.000001,6.961303e-08,0.000031
3197,CCC1=NN=NN1CC(=O)NC2CCN(CC2)C(C)=O,5.072451,0.674065,3.751284,6.393619,0.000008,4.040000e-07,0.000177
3198,CC=1C=C2N=CN(CC(O)CN3CCS(=O)CC3)C2=CC1C,5.241936,0.674083,3.920735,6.563138,0.000006,2.734399e-07,0.000120


### Use RDKit for basic druglikeness

A compound is considered to pass Lipinski’s Rule of Five (Ro5)  if it meets all four criteria:

- Molecular weight (MW) ≤ 500
- LogP ≤ 5(RDKit uses the Crippen cLogP)
- Hydrogen bond donors (HBD) ≤ 5
- Hydrogen bond acceptors (HBA) ≤ 10

These rules estimate membrane permeability and oral absorption potential.

In [8]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

# Calculate Lipinski's Rule of Five
def check_lipinski(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)
    return mw <= 500 and logp <= 5 and hbd <= 5 and hba <= 10

# Apply Lipinski filter
EnamineAntiviralsData_predicted['Lipinski_Pass'] = (
    EnamineAntiviralsData_predicted['SMILES'].apply(check_lipinski)
)

# Filter passing molecules
EnamineAntiviralsData_predicted_clean = (
    EnamineAntiviralsData_predicted[
        EnamineAntiviralsData_predicted['Lipinski_Pass'] == True
    ].reset_index(drop=True)
)

# --------------------------------------
# Print summary statistics
# --------------------------------------
total = len(EnamineAntiviralsData_predicted)
passed = EnamineAntiviralsData_predicted['Lipinski_Pass'].sum()
print(f"Lipinski-Passing Molecules: {passed} out of {total} ({passed/total*100:.2f}%)")

Lipinski-Passing Molecules: 3200 out of 3200 (100.00%)


,SMILES,pPotency_prediction,pPotency_std,pPotency_lower_95CI,pPotency_upper_95CI,IC50(M)_prediction,IC50(M)_lower_95CI,IC50(M)_upper_95CI,Lipinski_Pass
0,COCC(C)NC(=O)NC1=NN=C(S1)C2CC2,5.119991,0.674573,3.797828,6.442153,0.000008,3.612826e-07,0.000159,True
1,CN1C(=O)C=CN(CC(=O)NCC2CC2)C1=O,5.409013,0.674142,4.087694,6.730331,0.000004,1.860668e-07,0.000082,True
2,CCN(CC)CCNC(=O)C1=CC=CN=C1N2CCOCC2,4.906726,0.674204,3.585286,6.228166,0.000012,5.913362e-07,0.000260,True
3,CCC(CNC(=O)CC=1C(C)=NOC1C)N2CCCC2,4.879879,0.674231,3.558385,6.201372,0.000013,6.289675e-07,0.000276,True
4,CCC(C)(CNC(=O)CN1C=CC=CC1=O)N2CCOCC2,5.198782,0.674140,3.877469,6.520096,0.000006,3.019287e-07,0.000133,True
...,...,...,...,...,...,...,...,...,...
3195,CCOC=1N=CC=CC1NC(=O)NC2CCN3CCCCC23,5.240053,0.674091,3.918835,6.561272,0.000006,2.746173e-07,0.000121,True
3196,CC1=NN(CC(O)C=2C=CC=CC2)C(=O)C(C#N)=C1C,5.835830,0.674224,4.514350,7.157309,0.000001,6.961303e-08,0.000031,True
3197,CCC1=NN=NN1CC(=O)NC2CCN(CC2)C(C)=O,5.072451,0.674065,3.751284,6.393619,0.000008,4.040000e-07,0.000177,True
3198,CC=1C=C2N=CN(CC(O)CN3CCS(=O)CC3)C2=CC1C,5.241936,0.674083,3.920735,6.563138,0.000006,2.734399e-07,0.000120,True


### 1. Find toxicity data from using ADMET_AI

- https://github.com/swansonk14/admet_ai
- https://admet.ai.greenstonebio.com/

In [9]:
from admet_ai import ADMETModel

# Initialize model
model = ADMETModel()

# Get predictions for all SMILES
smiles_list = EnamineAntiviralsData_predicted['SMILES'].tolist()
predictions = model.predict(smiles=smiles_list)

# Check what predictions looks like
print("Predictions type:", type(predictions))
print("Predictions shape:", predictions.shape if hasattr(predictions, 'shape') else len(predictions))

# Reset indices before concatenating
predictions_reset = predictions.reset_index(drop=True)
original_reset = EnamineAntiviralsData_predicted.reset_index(drop=True)

# Concatenate with reset indices
EnamineAntiviralsData_predicted = pd.concat([original_reset, predictions_reset], axis=1)
EnamineAntiviralsData_predicted.head()

Loading pretrained parameter "encoder.encoder.0.cached_zero_vector".
Loading pretrained parameter "encoder.encoder.0.W_i.weight".
Loading pretrained parameter "encoder.encoder.0.W_h.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.bias".
Loading pretrained parameter "readout.1.weight".
Loading pretrained parameter "readout.1.bias".
Loading pretrained parameter "readout.4.weight".
Loading pretrained parameter "readout.4.bias".
Loading pretrained parameter "encoder.encoder.0.cached_zero_vector".
Loading pretrained parameter "encoder.encoder.0.W_i.weight".
Loading pretrained parameter "encoder.encoder.0.W_h.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.bias".
Loading pretrained parameter "readout.1.weight".
Loading pretrained parameter "readout.1.bias".
Loading pretrained parameter "readout.4.weight".
Loading pretrained parameter "readout.4.b

,SMILES,pPotency_prediction,pPotency_std,pPotency_lower_95CI,pPotency_upper_95CI,IC50(M)_prediction,IC50(M)_lower_95CI,IC50(M)_upper_95CI,Lipinski_Pass,molecular_weight,...,Caco2_Wang_drugbank_approved_percentile,Clearance_Hepatocyte_AZ_drugbank_approved_percentile,Clearance_Microsome_AZ_drugbank_approved_percentile,Half_Life_Obach_drugbank_approved_percentile,HydrationFreeEnergy_FreeSolv_drugbank_approved_percentile,LD50_Zhu_drugbank_approved_percentile,Lipophilicity_AstraZeneca_drugbank_approved_percentile,PPBR_AZ_drugbank_approved_percentile,Solubility_AqSolDB_drugbank_approved_percentile,VDss_Lombardo_drugbank_approved_percentile
0,COCC(C)NC(=O)NC1=NN=C(S1)C2CC2,5.119991,0.674573,3.797828,6.442153,0.000008,3.612826e-07,0.000159,True,256.331,...,87.592090,73.322993,71.074060,35.284994,26.017836,38.542071,57.386584,35.595192,54.711128,19.968980
1,CN1C(=O)C=CN(CC(=O)NCC2CC2)C1=O,5.409013,0.674142,4.087694,6.730331,0.000004,1.860668e-07,0.000082,True,237.259,...,97.014347,50.368360,41.295076,47.266382,8.763086,54.439705,19.193486,14.346646,85.963552,27.762699
2,CCN(CC)CCNC(=O)C1=CC=CN=C1N2CCOCC2,4.906726,0.674204,3.585286,6.228166,0.000012,5.913362e-07,0.000260,True,306.410,...,79.798371,56.921287,25.668864,74.563784,14.734393,47.150058,20.938348,3.722373,88.367584,94.222567
3,CCC(CNC(=O)CC=1C(C)=NOC1C)N2CCCC2,4.879879,0.674231,3.558385,6.201372,0.000013,6.289675e-07,0.000276,True,279.384,...,58.549826,39.976735,28.421869,48.856146,34.703373,62.815045,25.242342,8.181466,86.545173,59.170221
4,CCC(C)(CNC(=O)CN1C=CC=CC1=O)N2CCOCC2,5.198782,0.674140,3.877469,6.520096,0.000006,3.019287e-07,0.000133,True,307.394,...,65.529275,69.833269,20.589376,56.184568,17.836371,32.880962,21.170997,5.971307,88.018612,73.206669


### Keep only critical columns related to toxicity

ADMET_AI added 98 columns, now keeping all those columns related to toxicity

In [14]:
# Your original dataframe
EnamineAntiviralsData_predicted_wToxicity = EnamineAntiviralsData_predicted.copy()

# Toxicity-related columns
toxicity_cols = [
    "AMES", "hERG", "DILI", "ClinTox",
    "Carcinogens_Lagunin", "Skin_Reaction",
    
    # Stress response pathways (toxicity-related)
    "SR-ARE", "SR-ATAD5", "SR-HSE",
    "SR-MMP", "SR-p53",
    
    # Nuclear receptor toxicity
    "NR-AR-LBD", "NR-AR", "NR-AhR",
    "NR-Aromatase", "NR-ER-LBD", "NR-ER",
    "NR-PPAR-gamma",
    
    # Permeability/efflux interactions relevant to toxicity
    "BBB_Martins", "Pgp_Broccatelli", "Caco2_Wang",
    "HIA_Hou", "PAMPA_NCATS",
    
    # CYP-related metabolic liability (toxicity-relevant)
    "CYP1A2_Veith", "CYP2C19_Veith",
    "CYP2C9_Substrate_CarbonMangels", "CYP2C9_Veith",
    "CYP2D6_Substrate_CarbonMangels", "CYP2D6_Veith",
    "CYP3A4_Substrate_CarbonMangels", "CYP3A4_Veith",
    
    # Acute toxicity
    "LD50_Zhu",
    
    # Misc toxicity signals
    "Bioavailability_Ma",
]

# Required potency and SMILES columns
prediction_cols = [
    'SMILES', 
    'pPotency_prediction', 'pPotency_std',
    'pPotency_lower_95CI', 'pPotency_upper_95CI',
    'IC50(M)_prediction', 'IC50(M)_lower_95CI', 'IC50(M)_upper_95CI'
]

# Keep only columns that exist in the dataframe
cols_to_keep = prediction_cols + [c for c in toxicity_cols if c in EnamineAntiviralsData_predicted_wToxicity.columns]

# Filter dataframe
EnamineAntiviralsData_predicted_wToxicity = EnamineAntiviralsData_predicted_wToxicity[cols_to_keep].copy()
EnamineAntiviralsData_predicted_wToxicity

,SMILES,pPotency_prediction,pPotency_std,pPotency_lower_95CI,pPotency_upper_95CI,IC50(M)_prediction,IC50(M)_lower_95CI,IC50(M)_upper_95CI,AMES,hERG,...,CYP1A2_Veith,CYP2C19_Veith,CYP2C9_Substrate_CarbonMangels,CYP2C9_Veith,CYP2D6_Substrate_CarbonMangels,CYP2D6_Veith,CYP3A4_Substrate_CarbonMangels,CYP3A4_Veith,LD50_Zhu,Bioavailability_Ma
0,COCC(C)NC(=O)NC1=NN=C(S1)C2CC2,5.119991,0.674573,3.797828,6.442153,0.000008,3.612826e-07,0.000159,0.239500,0.016462,...,0.076049,0.350395,0.290878,0.134904,0.071786,0.002689,0.382473,0.005204,2.346986,0.945640
1,CN1C(=O)C=CN(CC(=O)NCC2CC2)C1=O,5.409013,0.674142,4.087694,6.730331,0.000004,1.860668e-07,0.000082,0.152839,0.008214,...,0.003674,0.034129,0.375321,0.005663,0.109176,0.002534,0.404691,0.004057,2.588685,0.914472
2,CCN(CC)CCNC(=O)C1=CC=CN=C1N2CCOCC2,4.906726,0.674204,3.585286,6.228166,0.000012,5.913362e-07,0.000260,0.638718,0.690349,...,0.091784,0.044804,0.168623,0.005473,0.320181,0.105699,0.264736,0.014986,2.486431,0.946715
3,CCC(CNC(=O)CC=1C(C)=NOC1C)N2CCCC2,4.879879,0.674231,3.558385,6.201372,0.000013,6.289675e-07,0.000276,0.121497,0.362352,...,0.003908,0.038184,0.042111,0.001958,0.348713,0.191350,0.608893,0.005841,2.715900,0.896890
4,CCC(C)(CNC(=O)CN1C=CC=CC1=O)N2CCOCC2,5.198782,0.674140,3.877469,6.520096,0.000006,3.019287e-07,0.000133,0.197948,0.221449,...,0.000397,0.065866,0.074823,0.006422,0.053373,0.042926,0.389752,0.044676,2.247248,0.823651
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3195,CCOC=1N=CC=CC1NC(=O)NC2CCN3CCCCC23,5.240053,0.674091,3.918835,6.561272,0.000006,2.746173e-07,0.000121,0.148568,0.533976,...,0.011429,0.025343,0.102413,0.006730,0.370298,0.274705,0.538261,0.003973,2.718897,0.851799
3196,CC1=NN(CC(O)C=2C=CC=CC2)C(=O)C(C#N)=C1C,5.835830,0.674224,4.514350,7.157309,0.000001,6.961303e-08,0.000031,0.171445,0.137801,...,0.197902,0.657578,0.424023,0.230949,0.065615,0.012771,0.524938,0.065732,2.299490,0.944819
3197,CCC1=NN=NN1CC(=O)NC2CCN(CC2)C(C)=O,5.072451,0.674065,3.751284,6.393619,0.000008,4.040000e-07,0.000177,0.453643,0.004518,...,0.000252,0.024954,0.027867,0.000396,0.021295,0.000457,0.583954,0.000585,2.285230,0.898917
3198,CC=1C=C2N=CN(CC(O)CN3CCS(=O)CC3)C2=CC1C,5.241936,0.674083,3.920735,6.563138,0.000006,2.734399e-07,0.000120,0.257713,0.787099,...,0.050628,0.107513,0.156510,0.031222,0.469366,0.305325,0.393582,0.061106,2.401208,0.801064


### Select list of non-toxic molecules

| Endpoint                                   | Meaning                   | Toxic if…   |
| ------------------------------------------ | ------------------------- | ----------- |
| **AMES**                                   | mutagenicity              | ≥ 0.5       |
| **hERG**                                   | cardiotoxicity            | ≥ 0.5       |
| **DILI**                                   | drug-induced liver injury | ≥ 0.5       |
| **ClinTox**                                | clinical toxicity         | ≥ 0.5       |
| **Carcinogens_Lagunin**                    | carcinogenicity           | ≥ 0.5       |
| **Skin_Reaction**                          | irritation                | ≥ 0.5       |
| **Stress response (SR-ARE, SR-HSE, etc.)** | cytotoxic stress          | ≥ 0.5       |
| **LD50_Zhu**                               | acute toxicity            | ≤ 300 mg/kg |


In [27]:
EnamineAntiviralsData_predicted_wToxicity["LD50_Zhu"].describe()

count    3200.000000
mean        2.448731
std         0.317056
min         1.252568
25%         2.239111
50%         2.422997
75%         2.637888
max         3.731548
Name: LD50_Zhu, dtype: float64

In [26]:
# Start from your cleaned ADMET dataframe
df = EnamineAntiviralsData_predicted_wToxicity.copy()

# --------------------------------------------------------------
# Create toxicity flags (basic + stress response + CYP liabilities)
# --------------------------------------------------------------

# Basic toxicity endpoints
df["flag_AMES"]    = (df["AMES"] >= 0.5).astype(int)
df["flag_hERG"]    = (df["hERG"] >= 0.5).astype(int)
df["flag_DILI"]    = (df["DILI"] >= 0.5).astype(int)
df["flag_ClinTox"] = (df["ClinTox"] >= 0.5).astype(int)
df["flag_Carcino"] = (df["Carcinogens_Lagunin"] >= 0.5).astype(int)
df["flag_Skin"]    = (df["Skin_Reaction"] >= 0.5).astype(int)
df["flag_LD50"]    = (df["LD50_Zhu"] <= 50).astype(int)

# Stress response endpoints
stressCols = ["SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53"]
for col in stressCols:
    if col in df.columns:
        df[f"flag_{col}"] = (df[col] >= 0.5).astype(int)

# CYP metabolic liabilities
cypCols = [
    "CYP1A2_Veith", "CYP2C19_Veith",
    "CYP2C9_Substrate_CarbonMangels", "CYP2C9_Veith",
    "CYP2D6_Substrate_CarbonMangels", "CYP2D6_Veith",
    "CYP3A4_Substrate_CarbonMangels", "CYP3A4_Veith"
]
for col in cypCols:
    if col in df.columns:
        df[f"flag_{col}"] = (df[col] >= 0.5).astype(int)

# --------------------------------------------------------------
# Final combined toxicity flag (ANY risk → 1, NO risk → 0)
# --------------------------------------------------------------
flagCols = [c for c in df.columns if c.startswith("flag_")]
df["Toxicity_Flag"] = (df[flagCols].sum(axis=1) > 0).astype(int)


# --------------------------------------------------------------
# Function: print summary of toxicity flags
# --------------------------------------------------------------
def printFlagSummary(df, flagCols):
    total = len(df)

    print("\n------------- Toxicity Flag Summary -------------")
    summary = df[flagCols].sum(axis=0).sort_values(ascending=False)
    for flag, count in summary.items():
        pct = (count / total) * 100
        print(f"{flag}: {count} flagged ({pct:.2f}%)")
    print("-------------------------------------------------\n")
    return summary


# Print toxicity summary
flagSummary = printFlagSummary(df, flagCols)


# --------------------------------------------------------------
# Keep only the final required output columns
# --------------------------------------------------------------
finalCols = [
    "SMILES",
    "pPotency_prediction", "pPotency_std",
    "pPotency_lower_95CI", "pPotency_upper_95CI",
    "IC50(M)_prediction", "IC50(M)_lower_95CI", "IC50(M)_upper_95CI",
    "Toxicity_Flag"
]

dfFinal = df[finalCols].copy()

# --------------------------------------------------------------
# Filter ONLY non-toxic molecules
# --------------------------------------------------------------
EnamineAntiviralsData_predicted_nonToxic = (
    dfFinal[dfFinal["Toxicity_Flag"] == 0]
    .reset_index(drop=True)
)

EnamineAntiviralsData_predicted_nonToxic


------------- Toxicity Flag Summary -------------
flag_LD50: 3200 flagged (100.00%)
flag_DILI: 1335 flagged (41.72%)
flag_CYP3A4_Substrate_CarbonMangels: 1334 flagged (41.69%)
flag_Skin: 1180 flagged (36.88%)
flag_CYP2C19_Veith: 734 flagged (22.94%)
flag_hERG: 409 flagged (12.78%)
flag_AMES: 390 flagged (12.19%)
flag_ClinTox: 383 flagged (11.97%)
flag_CYP1A2_Veith: 372 flagged (11.62%)
flag_CYP3A4_Veith: 361 flagged (11.28%)
flag_CYP2C9_Substrate_CarbonMangels: 292 flagged (9.12%)
flag_CYP2C9_Veith: 155 flagged (4.84%)
flag_CYP2D6_Veith: 135 flagged (4.22%)
flag_CYP2D6_Substrate_CarbonMangels: 60 flagged (1.88%)
flag_SR-ARE: 44 flagged (1.38%)
flag_SR-MMP: 42 flagged (1.31%)
flag_Carcino: 37 flagged (1.16%)
flag_SR-HSE: 2 flagged (0.06%)
flag_SR-p53: 0 flagged (0.00%)
flag_SR-ATAD5: 0 flagged (0.00%)
-------------------------------------------------



,SMILES,pPotency_prediction,pPotency_std,pPotency_lower_95CI,pPotency_upper_95CI,IC50(M)_prediction,IC50(M)_lower_95CI,IC50(M)_upper_95CI,Toxicity_Flag


### Check toxicty for Lifechemicals AntiviralsData set

In [30]:
LCAntiviralsData_predicted = pd.read_csv(saveDir + 'AllVirus_wART_LCAntiviralsData_predicted_all.csv')
max_val = LCAntiviralsData_predicted['pPotency_prediction'].max()
min_val = LCAntiviralsData_predicted['pPotency_prediction'].min()
print(f"pPotency_prediction range: {min_val:.3f} → {max_val:.3f}")
LCAntiviralsData_predicted

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/AllVirus/AllVirus_wART_LCAntiviralsData_predicted_all.csv'

### Use RDKit for basic druglikeness

A compound is considered to pass Lipinski’s Rule of Five (Ro5)  if it meets all four criteria:

- Molecular weight (MW) ≤ 500
- LogP ≤ 5(RDKit uses the Crippen cLogP)
- Hydrogen bond donors (HBD) ≤ 5
- Hydrogen bond acceptors (HBA) ≤ 10

These rules estimate membrane permeability and oral absorption potential.

In [8]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

# Calculate Lipinski's Rule of Five
def check_lipinski(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)
    return mw <= 500 and logp <= 5 and hbd <= 5 and hba <= 10

# Apply Lipinski filter
LCAntiviralsData_predicted['Lipinski_Pass'] = (
    LCAntiviralsData_predicted['SMILES'].apply(check_lipinski)
)

# Filter passing molecules
LCAntiviralsData_predicted_clean = (
    LCAntiviralsData_predicted[
        LCAntiviralsData_predicted['Lipinski_Pass'] == True
    ].reset_index(drop=True)
)

# --------------------------------------
# Print summary statistics
# --------------------------------------
total = len(LCAntiviralsData_predicted)
passed = LCAntiviralsData_predicted['Lipinski_Pass'].sum()
print(f"Lipinski-Passing Molecules: {passed} out of {total} ({passed/total*100:.2f}%)")

Lipinski-Passing Molecules: 3200 out of 3200 (100.00%)


,SMILES,pPotency_prediction,pPotency_std,pPotency_lower_95CI,pPotency_upper_95CI,IC50(M)_prediction,IC50(M)_lower_95CI,IC50(M)_upper_95CI,Lipinski_Pass
0,COCC(C)NC(=O)NC1=NN=C(S1)C2CC2,5.119991,0.674573,3.797828,6.442153,0.000008,3.612826e-07,0.000159,True
1,CN1C(=O)C=CN(CC(=O)NCC2CC2)C1=O,5.409013,0.674142,4.087694,6.730331,0.000004,1.860668e-07,0.000082,True
2,CCN(CC)CCNC(=O)C1=CC=CN=C1N2CCOCC2,4.906726,0.674204,3.585286,6.228166,0.000012,5.913362e-07,0.000260,True
3,CCC(CNC(=O)CC=1C(C)=NOC1C)N2CCCC2,4.879879,0.674231,3.558385,6.201372,0.000013,6.289675e-07,0.000276,True
4,CCC(C)(CNC(=O)CN1C=CC=CC1=O)N2CCOCC2,5.198782,0.674140,3.877469,6.520096,0.000006,3.019287e-07,0.000133,True
...,...,...,...,...,...,...,...,...,...
3195,CCOC=1N=CC=CC1NC(=O)NC2CCN3CCCCC23,5.240053,0.674091,3.918835,6.561272,0.000006,2.746173e-07,0.000121,True
3196,CC1=NN(CC(O)C=2C=CC=CC2)C(=O)C(C#N)=C1C,5.835830,0.674224,4.514350,7.157309,0.000001,6.961303e-08,0.000031,True
3197,CCC1=NN=NN1CC(=O)NC2CCN(CC2)C(C)=O,5.072451,0.674065,3.751284,6.393619,0.000008,4.040000e-07,0.000177,True
3198,CC=1C=C2N=CN(CC(O)CN3CCS(=O)CC3)C2=CC1C,5.241936,0.674083,3.920735,6.563138,0.000006,2.734399e-07,0.000120,True


### 1. Find toxicity data from using ADMET_AI

- https://github.com/swansonk14/admet_ai
- https://admet.ai.greenstonebio.com/

In [9]:
from admet_ai import ADMETModel

# Initialize model
model = ADMETModel()

# Get predictions for all SMILES
smiles_list = LCAntiviralsData_predicted['SMILES'].tolist()
predictions = model.predict(smiles=smiles_list)

# Check what predictions looks like
print("Predictions type:", type(predictions))
print("Predictions shape:", predictions.shape if hasattr(predictions, 'shape') else len(predictions))

# Reset indices before concatenating
predictions_reset = predictions.reset_index(drop=True)
original_reset = LCAntiviralsData_predicted.reset_index(drop=True)

# Concatenate with reset indices
LCAntiviralsData_predicted = pd.concat([original_reset, predictions_reset], axis=1)
LCAntiviralsData_predicted.head()

Loading pretrained parameter "encoder.encoder.0.cached_zero_vector".
Loading pretrained parameter "encoder.encoder.0.W_i.weight".
Loading pretrained parameter "encoder.encoder.0.W_h.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.bias".
Loading pretrained parameter "readout.1.weight".
Loading pretrained parameter "readout.1.bias".
Loading pretrained parameter "readout.4.weight".
Loading pretrained parameter "readout.4.bias".
Loading pretrained parameter "encoder.encoder.0.cached_zero_vector".
Loading pretrained parameter "encoder.encoder.0.W_i.weight".
Loading pretrained parameter "encoder.encoder.0.W_h.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.bias".
Loading pretrained parameter "readout.1.weight".
Loading pretrained parameter "readout.1.bias".
Loading pretrained parameter "readout.4.weight".
Loading pretrained parameter "readout.4.b

,SMILES,pPotency_prediction,pPotency_std,pPotency_lower_95CI,pPotency_upper_95CI,IC50(M)_prediction,IC50(M)_lower_95CI,IC50(M)_upper_95CI,Lipinski_Pass,molecular_weight,...,Caco2_Wang_drugbank_approved_percentile,Clearance_Hepatocyte_AZ_drugbank_approved_percentile,Clearance_Microsome_AZ_drugbank_approved_percentile,Half_Life_Obach_drugbank_approved_percentile,HydrationFreeEnergy_FreeSolv_drugbank_approved_percentile,LD50_Zhu_drugbank_approved_percentile,Lipophilicity_AstraZeneca_drugbank_approved_percentile,PPBR_AZ_drugbank_approved_percentile,Solubility_AqSolDB_drugbank_approved_percentile,VDss_Lombardo_drugbank_approved_percentile
0,COCC(C)NC(=O)NC1=NN=C(S1)C2CC2,5.119991,0.674573,3.797828,6.442153,0.000008,3.612826e-07,0.000159,True,256.331,...,87.592090,73.322993,71.074060,35.284994,26.017836,38.542071,57.386584,35.595192,54.711128,19.968980
1,CN1C(=O)C=CN(CC(=O)NCC2CC2)C1=O,5.409013,0.674142,4.087694,6.730331,0.000004,1.860668e-07,0.000082,True,237.259,...,97.014347,50.368360,41.295076,47.266382,8.763086,54.439705,19.193486,14.346646,85.963552,27.762699
2,CCN(CC)CCNC(=O)C1=CC=CN=C1N2CCOCC2,4.906726,0.674204,3.585286,6.228166,0.000012,5.913362e-07,0.000260,True,306.410,...,79.798371,56.921287,25.668864,74.563784,14.734393,47.150058,20.938348,3.722373,88.367584,94.222567
3,CCC(CNC(=O)CC=1C(C)=NOC1C)N2CCCC2,4.879879,0.674231,3.558385,6.201372,0.000013,6.289675e-07,0.000276,True,279.384,...,58.549826,39.976735,28.421869,48.856146,34.703373,62.815045,25.242342,8.181466,86.545173,59.170221
4,CCC(C)(CNC(=O)CN1C=CC=CC1=O)N2CCOCC2,5.198782,0.674140,3.877469,6.520096,0.000006,3.019287e-07,0.000133,True,307.394,...,65.529275,69.833269,20.589376,56.184568,17.836371,32.880962,21.170997,5.971307,88.018612,73.206669


### Keep only critical columns related to toxicity

ADMET_AI added 98 columns, now keeping all those columns related to toxicity

In [14]:
# Your original dataframe
LCAntiviralsData_predicted_wToxicity = LCAntiviralsData_predicted.copy()

# Toxicity-related columns
toxicity_cols = [
    "AMES", "hERG", "DILI", "ClinTox",
    "Carcinogens_Lagunin", "Skin_Reaction",
    
    # Stress response pathways (toxicity-related)
    "SR-ARE", "SR-ATAD5", "SR-HSE",
    "SR-MMP", "SR-p53",
    
    # Nuclear receptor toxicity
    "NR-AR-LBD", "NR-AR", "NR-AhR",
    "NR-Aromatase", "NR-ER-LBD", "NR-ER",
    "NR-PPAR-gamma",
    
    # Permeability/efflux interactions relevant to toxicity
    "BBB_Martins", "Pgp_Broccatelli", "Caco2_Wang",
    "HIA_Hou", "PAMPA_NCATS",
    
    # CYP-related metabolic liability (toxicity-relevant)
    "CYP1A2_Veith", "CYP2C19_Veith",
    "CYP2C9_Substrate_CarbonMangels", "CYP2C9_Veith",
    "CYP2D6_Substrate_CarbonMangels", "CYP2D6_Veith",
    "CYP3A4_Substrate_CarbonMangels", "CYP3A4_Veith",
    
    # Acute toxicity
    "LD50_Zhu",
    
    # Misc toxicity signals
    "Bioavailability_Ma",
]

# Required potency and SMILES columns
prediction_cols = [
    'SMILES', 
    'pPotency_prediction', 'pPotency_std',
    'pPotency_lower_95CI', 'pPotency_upper_95CI',
    'IC50(M)_prediction', 'IC50(M)_lower_95CI', 'IC50(M)_upper_95CI'
]

# Keep only columns that exist in the dataframe
cols_to_keep = prediction_cols + [c for c in toxicity_cols if c in LCAntiviralsData_predicted_wToxicity.columns]

# Filter dataframe
LCAntiviralsData_predicted_wToxicity = LCAntiviralsData_predicted_wToxicity[cols_to_keep].copy()
LCAntiviralsData_predicted_wToxicity

,SMILES,pPotency_prediction,pPotency_std,pPotency_lower_95CI,pPotency_upper_95CI,IC50(M)_prediction,IC50(M)_lower_95CI,IC50(M)_upper_95CI,AMES,hERG,...,CYP1A2_Veith,CYP2C19_Veith,CYP2C9_Substrate_CarbonMangels,CYP2C9_Veith,CYP2D6_Substrate_CarbonMangels,CYP2D6_Veith,CYP3A4_Substrate_CarbonMangels,CYP3A4_Veith,LD50_Zhu,Bioavailability_Ma
0,COCC(C)NC(=O)NC1=NN=C(S1)C2CC2,5.119991,0.674573,3.797828,6.442153,0.000008,3.612826e-07,0.000159,0.239500,0.016462,...,0.076049,0.350395,0.290878,0.134904,0.071786,0.002689,0.382473,0.005204,2.346986,0.945640
1,CN1C(=O)C=CN(CC(=O)NCC2CC2)C1=O,5.409013,0.674142,4.087694,6.730331,0.000004,1.860668e-07,0.000082,0.152839,0.008214,...,0.003674,0.034129,0.375321,0.005663,0.109176,0.002534,0.404691,0.004057,2.588685,0.914472
2,CCN(CC)CCNC(=O)C1=CC=CN=C1N2CCOCC2,4.906726,0.674204,3.585286,6.228166,0.000012,5.913362e-07,0.000260,0.638718,0.690349,...,0.091784,0.044804,0.168623,0.005473,0.320181,0.105699,0.264736,0.014986,2.486431,0.946715
3,CCC(CNC(=O)CC=1C(C)=NOC1C)N2CCCC2,4.879879,0.674231,3.558385,6.201372,0.000013,6.289675e-07,0.000276,0.121497,0.362352,...,0.003908,0.038184,0.042111,0.001958,0.348713,0.191350,0.608893,0.005841,2.715900,0.896890
4,CCC(C)(CNC(=O)CN1C=CC=CC1=O)N2CCOCC2,5.198782,0.674140,3.877469,6.520096,0.000006,3.019287e-07,0.000133,0.197948,0.221449,...,0.000397,0.065866,0.074823,0.006422,0.053373,0.042926,0.389752,0.044676,2.247248,0.823651
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3195,CCOC=1N=CC=CC1NC(=O)NC2CCN3CCCCC23,5.240053,0.674091,3.918835,6.561272,0.000006,2.746173e-07,0.000121,0.148568,0.533976,...,0.011429,0.025343,0.102413,0.006730,0.370298,0.274705,0.538261,0.003973,2.718897,0.851799
3196,CC1=NN(CC(O)C=2C=CC=CC2)C(=O)C(C#N)=C1C,5.835830,0.674224,4.514350,7.157309,0.000001,6.961303e-08,0.000031,0.171445,0.137801,...,0.197902,0.657578,0.424023,0.230949,0.065615,0.012771,0.524938,0.065732,2.299490,0.944819
3197,CCC1=NN=NN1CC(=O)NC2CCN(CC2)C(C)=O,5.072451,0.674065,3.751284,6.393619,0.000008,4.040000e-07,0.000177,0.453643,0.004518,...,0.000252,0.024954,0.027867,0.000396,0.021295,0.000457,0.583954,0.000585,2.285230,0.898917
3198,CC=1C=C2N=CN(CC(O)CN3CCS(=O)CC3)C2=CC1C,5.241936,0.674083,3.920735,6.563138,0.000006,2.734399e-07,0.000120,0.257713,0.787099,...,0.050628,0.107513,0.156510,0.031222,0.469366,0.305325,0.393582,0.061106,2.401208,0.801064


### Select list of non-toxic molecules

| Endpoint                                   | Meaning                   | Toxic if…   |
| ------------------------------------------ | ------------------------- | ----------- |
| **AMES**                                   | mutagenicity              | ≥ 0.5       |
| **hERG**                                   | cardiotoxicity            | ≥ 0.5       |
| **DILI**                                   | drug-induced liver injury | ≥ 0.5       |
| **ClinTox**                                | clinical toxicity         | ≥ 0.5       |
| **Carcinogens_Lagunin**                    | carcinogenicity           | ≥ 0.5       |
| **Skin_Reaction**                          | irritation                | ≥ 0.5       |
| **Stress response (SR-ARE, SR-HSE, etc.)** | cytotoxic stress          | ≥ 0.5       |
| **LD50_Zhu**                               | acute toxicity            | ≤ 300 mg/kg |


In [27]:
LCAntiviralsData_predicted_wToxicity["LD50_Zhu"].describe()

count    3200.000000
mean        2.448731
std         0.317056
min         1.252568
25%         2.239111
50%         2.422997
75%         2.637888
max         3.731548
Name: LD50_Zhu, dtype: float64

In [26]:
# Start from your cleaned ADMET dataframe
df = LCAntiviralsData_predicted_wToxicity.copy()

# --------------------------------------------------------------
# Create toxicity flags (basic + stress response + CYP liabilities)
# --------------------------------------------------------------

# Basic toxicity endpoints
df["flag_AMES"]    = (df["AMES"] >= 0.5).astype(int)
df["flag_hERG"]    = (df["hERG"] >= 0.5).astype(int)
df["flag_DILI"]    = (df["DILI"] >= 0.5).astype(int)
df["flag_ClinTox"] = (df["ClinTox"] >= 0.5).astype(int)
df["flag_Carcino"] = (df["Carcinogens_Lagunin"] >= 0.5).astype(int)
df["flag_Skin"]    = (df["Skin_Reaction"] >= 0.5).astype(int)
df["flag_LD50"]    = (df["LD50_Zhu"] <= 50).astype(int)

# Stress response endpoints
stressCols = ["SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53"]
for col in stressCols:
    if col in df.columns:
        df[f"flag_{col}"] = (df[col] >= 0.5).astype(int)

# CYP metabolic liabilities
cypCols = [
    "CYP1A2_Veith", "CYP2C19_Veith",
    "CYP2C9_Substrate_CarbonMangels", "CYP2C9_Veith",
    "CYP2D6_Substrate_CarbonMangels", "CYP2D6_Veith",
    "CYP3A4_Substrate_CarbonMangels", "CYP3A4_Veith"
]
for col in cypCols:
    if col in df.columns:
        df[f"flag_{col}"] = (df[col] >= 0.5).astype(int)

# --------------------------------------------------------------
# Final combined toxicity flag (ANY risk → 1, NO risk → 0)
# --------------------------------------------------------------
flagCols = [c for c in df.columns if c.startswith("flag_")]
df["Toxicity_Flag"] = (df[flagCols].sum(axis=1) > 0).astype(int)


# --------------------------------------------------------------
# Function: print summary of toxicity flags
# --------------------------------------------------------------
def printFlagSummary(df, flagCols):
    total = len(df)

    print("\n------------- Toxicity Flag Summary -------------")
    summary = df[flagCols].sum(axis=0).sort_values(ascending=False)
    for flag, count in summary.items():
        pct = (count / total) * 100
        print(f"{flag}: {count} flagged ({pct:.2f}%)")
    print("-------------------------------------------------\n")
    return summary


# Print toxicity summary
flagSummary = printFlagSummary(df, flagCols)


# --------------------------------------------------------------
# Keep only the final required output columns
# --------------------------------------------------------------
finalCols = [
    "SMILES",
    "pPotency_prediction", "pPotency_std",
    "pPotency_lower_95CI", "pPotency_upper_95CI",
    "IC50(M)_prediction", "IC50(M)_lower_95CI", "IC50(M)_upper_95CI",
    "Toxicity_Flag"
]

dfFinal = df[finalCols].copy()

# --------------------------------------------------------------
# Filter ONLY non-toxic molecules
# --------------------------------------------------------------
LCAntiviralsData_predicted_nonToxic = (
    dfFinal[dfFinal["Toxicity_Flag"] == 0]
    .reset_index(drop=True)
)

LCAntiviralsData_predicted_nonToxic


------------- Toxicity Flag Summary -------------
flag_LD50: 3200 flagged (100.00%)
flag_DILI: 1335 flagged (41.72%)
flag_CYP3A4_Substrate_CarbonMangels: 1334 flagged (41.69%)
flag_Skin: 1180 flagged (36.88%)
flag_CYP2C19_Veith: 734 flagged (22.94%)
flag_hERG: 409 flagged (12.78%)
flag_AMES: 390 flagged (12.19%)
flag_ClinTox: 383 flagged (11.97%)
flag_CYP1A2_Veith: 372 flagged (11.62%)
flag_CYP3A4_Veith: 361 flagged (11.28%)
flag_CYP2C9_Substrate_CarbonMangels: 292 flagged (9.12%)
flag_CYP2C9_Veith: 155 flagged (4.84%)
flag_CYP2D6_Veith: 135 flagged (4.22%)
flag_CYP2D6_Substrate_CarbonMangels: 60 flagged (1.88%)
flag_SR-ARE: 44 flagged (1.38%)
flag_SR-MMP: 42 flagged (1.31%)
flag_Carcino: 37 flagged (1.16%)
flag_SR-HSE: 2 flagged (0.06%)
flag_SR-p53: 0 flagged (0.00%)
flag_SR-ATAD5: 0 flagged (0.00%)
-------------------------------------------------



,SMILES,pPotency_prediction,pPotency_std,pPotency_lower_95CI,pPotency_upper_95CI,IC50(M)_prediction,IC50(M)_lower_95CI,IC50(M)_upper_95CI,Toxicity_Flag


### 2. Use ADMET3.0 web server

https://admetlab3.scbdd.com/server/screening

In [24]:
EnamineAntiviralsData_wToxicity_4rm_ADMET_webServer = pd.read_csv(dataDir + "Toxicity/ADMET_webServer.csv")
EnamineAntiviralsData_wToxicity_4rm_ADMET_webServer

,raw_smiles,smiles,MW,Vol,Dense,nHA,nHD,TPSA,nRot,nRing,...,Acute_Aquatic_Toxicity,FAF-Drugs4 Rule,Genotoxic_Carcinogenicity_Mutagenicity,Aggregators,Fluc,Blue_fluorescence,Green_fluorescence,Reactive,Other_assay_interference,Promiscuous
0,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)N(C)CC=3C=CC=CC3,COCCn1c(N(C)Cc2ccccc2)nc2c1c(=O)[nH]c(=O)n2C,343.16,339.818102,1.009834,8,1,85.15,6,3,...,['-'],"[(3, 4, 5, 15, 16, 17)]","[(5, 6)]",0.018,0.000,0.038,0.371,0.004,0.237,0.010
1,COCCN1C(=O)NC(=O)C(=C1N)N(CC=2C=CC=CC2)C(C)=O,COCCn1c(N)c(N(Cc2ccccc2)C(C)=O)c(=O)[nH]c1=O,332.15,328.872042,1.009967,8,3,110.42,7,2,...,['-'],['-'],"[(5, 6), (5, 6), (5,), (7, 8)]",0.010,0.000,0.102,0.330,0.005,0.445,0.026
2,COCCN1C=C(C=CC1=O)NC(=O)C2C(C=C(C)C)C2(C)C,COCCn1cc(NC(=O)C2C(C=C(C)C)C2(C)C)ccc1=O,318.19,337.953182,0.941521,5,1,60.33,7,2,...,['-'],['-'],"[(6, 7)]",0.023,0.001,0.047,0.174,0.003,0.190,0.007
3,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)NN=CC=3C=CC=CC3,COCCn1c(NN=Cc2ccccc2)nc2c1c(=O)[nH]c(=O)n2C,342.14,330.882419,1.034023,9,2,106.30,6,3,...,['-'],"[(8, 7, 6), (3, 4, 5, 15, 16, 17), (8, 7)]","[(5, 6)]",0.451,0.754,0.055,0.677,0.004,0.396,0.008
4,COC=1C=CC(OC)=C(C1)C(O)CN2C(C)=NC=3C=CC=CC32,COc1ccc(OC)c(C(O)Cn2c(C)nc3ccccc32)c1,312.15,324.123806,0.963058,5,1,56.51,5,3,...,['-'],"[(11, 12, 13, 15, 16, 21), (12, 21, 20, 19, 18...",['-'],0.101,0.075,0.056,0.093,0.005,0.100,0.034
5,CCC=1C=C(CC)N(N1)C2=NC3=C(C(=O)NC(=O)N3C)N2CCOC,CCc1cc(CC)n(-c2nc3c(c(=O)[nH]c(=O)n3C)n2CCOC)n1,346.18,336.155336,1.029822,9,1,99.73,6,3,...,['-'],"[(20, 19, 8, 9, 10, 11)]",['-'],0.009,0.001,0.148,0.156,0.006,0.219,0.038
6,CCOCCN1C(=NC2=C1C(=O)NC(=O)N2C)N3N=C(C)C=C3C,CCOCCn1c(-n2nc(C)cc2C)nc2c1c(=O)[nH]c(=O)n2C,332.16,318.859351,1.041713,9,1,99.73,5,3,...,['-'],"[(4, 5, 6, 14, 15, 16)]",['-'],0.008,0.000,0.074,0.199,0.004,0.454,0.023
7,CC=1C=CC=C(N1)NC(=O)NC2CCCN(CC(F)(F)F)C2=O,Cc1cccc(NC(=O)NC2CCCN(CC(F)(F)F)C2=O)n1,330.13,300.175213,1.099791,6,2,74.33,6,2,...,"[(15, 16, 17)]",['-'],"[(5, 6)]",0.016,0.003,0.032,0.192,0.001,0.274,0.006
8,CC1=CC=2N=CN(CC(O)CN3C(=O)NC(C)(C)C3=O)C2C=C1C,Cc1cc2ncn(CC(O)CN3C(=O)NC(C)(C)C3=O)c2cc1C,330.17,331.457800,0.996115,7,2,87.46,4,3,...,['-'],"[(7, 6, 5, 4, 3, 20), (4, 3, 2, 1, 22, 21, 20)]",['-'],0.029,0.000,0.223,0.476,0.002,0.016,0.001
9,COC=1C=CC(=CN1)NC(=O)NCCC(=O)N2CC(C)OC(C)C2,COc1ccc(NC(=O)NCCC(=O)N2CC(C)OC(C)C2)cn1,336.18,334.144960,1.006090,8,2,92.79,8,2,...,['-'],"[(7, 9, 10, 11, 12, 14, 15, 16, 18, 19)]","[(5, 6)]",0.042,0.001,0.025,0.423,0.001,0.116,0.000


In [33]:
#EnamineAntiviralsData_wToxicity_4rm_ADMETai_webServerAPI = EnamineAntiviralsData_predicted.copy()
EnamineAntiviralsData_wToxicity_4rm_ADMETai_webServer = pd.read_csv(dataDir + "Toxicity/ADMETai_webServer.csv")
EnamineAntiviralsData_wToxicity_4rm_ADMETai_webServer

,smiles,molecular_weight,logP,hydrogen_bond_acceptors,hydrogen_bond_donors,Lipinski,QED,stereo_centers,tpsa,AMES,...,Caco2_Wang_drugbank_approved_percentile,Clearance_Hepatocyte_AZ_drugbank_approved_percentile,Clearance_Microsome_AZ_drugbank_approved_percentile,Half_Life_Obach_drugbank_approved_percentile,HydrationFreeEnergy_FreeSolv_drugbank_approved_percentile,LD50_Zhu_drugbank_approved_percentile,Lipophilicity_AstraZeneca_drugbank_approved_percentile,PPBR_AZ_drugbank_approved_percentile,Solubility_AqSolDB_drugbank_approved_percentile,VDss_Lombardo_drugbank_approved_percentile
0,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)N(C)CC=3C=CC=CC3,343.387,0.70610,7,1,4.0,0.712301,0,85.15,0.357161,...,70.453664,72.857697,61.031408,32.531989,26.289259,63.086468,52.811167,56.417216,34.625824,26.134161
1,COCCN1C(=O)NC(=O)C(=C1N)N(CC=2C=CC=CC2)C(C)=O,332.360,0.31830,6,2,4.0,0.791264,0,110.42,0.366818,...,41.372625,59.092672,28.576968,8.724312,5.583560,34.082978,29.042264,32.376890,43.117487,51.841799
2,COCCN1C=C(C=CC1=O)NC(=O)C2C(C=C(C)C)C2(C)C,318.417,2.67160,4,1,4.0,0.820409,2,60.33,0.312303,...,97.324544,89.220628,80.069794,55.874370,58.898798,59.247770,75.222955,45.715394,39.666537,16.518030
3,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)NN=CC=3C=CC=CC3,342.359,0.51570,8,2,4.0,0.501558,0,106.30,0.604157,...,45.560295,54.672354,39.744087,38.037999,12.330361,46.452113,45.986817,54.749903,23.768903,18.844513
4,COC=1C=CC(OC)=C(C1)C(O)CN2C(C)=NC=3C=CC=CC32,312.369,3.09552,5,1,4.0,0.786339,1,56.51,0.280445,...,73.284219,52.151997,70.802637,19.426134,28.887166,45.133773,59.674292,58.200853,49.360217,11.050795
5,CCC=1C=C(CC)N(N1)C2=NC3=C(C(=O)NC(=O)N3C)N2CCOC,346.391,0.38020,8,1,4.0,0.695474,0,99.73,0.340428,...,71.461807,25.397441,38.309422,6.397829,30.709577,55.486623,37.805351,46.141915,49.360217,58.976347
6,CCOCCN1C(=NC2=C1C(=O)NC(=O)N2C)N3N=C(C)C=C3C,332.364,0.26234,8,1,4.0,0.675786,0,99.73,0.221195,...,64.637456,17.720047,34.664599,6.940675,24.621946,60.527336,28.770841,39.433889,62.466072,34.470725
7,CC=1C=CC=C(N1)NC(=O)NC2CCCN(CC(F)(F)F)C2=O,330.310,2.06482,3,2,4.0,0.890826,1,74.33,0.304816,...,58.821249,31.678945,28.615743,20.434277,31.252423,67.390461,52.345870,36.758434,56.998837,59.441644
8,CC1=CC=2N=CN(CC(O)CN3C(=O)NC(C)(C)C3=O)C2C=C1C,330.388,1.34444,5,2,4.0,0.828411,1,87.46,0.137849,...,69.872043,24.699496,53.625436,44.125630,6.979449,45.133773,54.866227,48.701047,53.935634,6.514153
9,COC=1C=CC(=CN1)NC(=O)NCCC(=O)N2CC(C)OC(C)C2,336.392,1.23760,5,2,4.0,0.843734,2,92.79,0.371379,...,74.486235,36.293137,32.725863,3.761148,14.036448,40.868554,44.707251,21.015898,77.200465,46.529663


### Check common columns

In [36]:
common_cols = set(EnamineAntiviralsData_wToxicity_4rm_ADMET_webServer.columns) & set(EnamineAntiviralsData_wToxicity_4rm_ADMETai_webServer.columns)
print(f"Number of common columns: {len(common_cols)}")
print(f"Common columns: {common_cols}")

Number of common columns: 18
Common columns: {'NR-Aromatase', 'NR-PPAR-gamma', 'SR-ATAD5', 'SR-MMP', 'NR-ER', 'NR-AR', 'Lipinski', 'logP', 'SR-ARE', 'NR-ER-LBD', 'SR-p53', 'smiles', 'DILI', 'QED', 'hERG', 'NR-AhR', 'NR-AR-LBD', 'SR-HSE'}


### 3. Using DeepChem for Tox21 Predictions

3.1 Use DeepChem's built-in Tox21 predictions

In [10]:
import deepchem as dc
import pandas as pd
import numpy as np
from rdkit import Chem

def predict_tox21_manual(smiles_list):
    """
    Predict Tox21 without using load_tox21 (which has bugs)
    """
    # Download and load Tox21 data directly
    import urllib.request
    import os
    
    print("Downloading Tox21 dataset...")
    toxicitydataDir = dataDir + '/tox21_data'
    os.makedirs(toxicitydataDir, exist_ok=True)
    
    dataset_file = os.path.join(toxicitydataDir, 'tox21.csv')
    
    if not os.path.exists(dataset_file):
        url = 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz'
        urllib.request.urlretrieve(url, dataset_file + '.gz')
        import gzip
        import shutil
        with gzip.open(dataset_file + '.gz', 'rb') as f_in:
            with open(dataset_file, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
    
    # Load the CSV
    print("Loading Tox21 data from CSV...")
    df = pd.read_csv(dataset_file)
    
    # Tox21 tasks
    tasks = ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 
             'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 
             'SR-HSE', 'SR-MMP', 'SR-p53']
    
    print(f"Dataset shape: {df.shape}")
    print(f"Tasks: {tasks}")
    
    # Featurize training data
    print("Featurizing training data...")
    featurizer = dc.feat.CircularFingerprint(size=1024)
    
    # Initialize lists to collect data
    train_features = []
    train_labels = []
    
    for idx, row in df.iterrows():
        try:
            smiles = row['smiles']
            feat = featurizer.featurize([smiles])[0]
            
            # Check if featurization was successful
            if feat is not None and len(feat) == 1024:  # Ensure correct size
                # Convert to 1D array if needed
                feat_array = np.array(feat).flatten()
                
                if not np.isnan(feat_array).any():
                    train_features.append(feat_array)
                    # Get labels for all tasks
                    labels = [float(row[task]) if pd.notna(row[task]) else 0.0 for task in tasks]
                    train_labels.append(labels)
                
        except Exception as e:
            continue
        
        if (idx + 1) % 1000 == 0:
            print(f"Processed {idx + 1}/{len(df)} molecules...")
    
    print(f"Successfully featurized {len(train_features)} training molecules")
    
    # Convert to arrays - stack them properly
    X_train = np.vstack(train_features)  # Use vstack instead of array
    y_train = np.array(train_labels)
    
    print(f"Training data shape: X={X_train.shape}, y={y_train.shape}")
    
    # Create dataset
    train_dataset = dc.data.NumpyDataset(X=X_train, y=y_train)
    
    # Train model - FIXED VERSION
    print("Training model...")
    
    # Use sklearn-based model instead (more stable)
    from sklearn.ensemble import RandomForestClassifier
    
    # Train a separate model for each task
    models = []
    for task_idx, task_name in enumerate(tasks):
        print(f"  Training {task_name}...")
        y_task = y_train[:, task_idx]
        
        # Only train on samples with valid labels
        valid_mask = ~np.isnan(y_task)
        X_valid = X_train[valid_mask]
        y_valid = y_task[valid_mask].astype(int)
        
        model = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_valid, y_valid)
        models.append(model)
    
    # Now predict on your molecules
    print(f"\nFeaturizing {len(smiles_list)} query molecules...")
    query_features = []
    valid_query_indices = []
    
    for i, smiles in enumerate(smiles_list):
        try:
            feat = featurizer.featurize([smiles])[0]
            if feat is not None and len(feat) == 1024:
                feat_array = np.array(feat).flatten()
                if not np.isnan(feat_array).any():
                    query_features.append(feat_array)
                    valid_query_indices.append(i)
        except:
            continue
    
    print(f"Successfully featurized {len(query_features)}/{len(smiles_list)} query molecules")
    
    # Predict
    X_query = np.vstack(query_features)  # Use vstack
    
    print("Making predictions...")
    predictions = []
    for task_idx, model in enumerate(models):
        # Get probability of positive class
        pred_proba = model.predict_proba(X_query)[:, 1]
        predictions.append(pred_proba)
    
    # Transpose to get (n_samples, n_tasks) shape
    predictions = np.array(predictions).T
    
    # Create results DataFrame
    results = pd.DataFrame(predictions, columns=tasks, index=valid_query_indices)
    
    return results, tasks

# Run
print("Starting Tox21 predictions with DeepChem...\n")
EnamineAntiviralsData_wToxicity_deepchem, tox21_tasks = predict_tox21_manual(
    EnamineAntiviralsData_predicted['SMILES'].tolist()
)

print("\nDeepChem Tox21 predictions:")
EnamineAntiviralsData_wToxicity_deepchem.head()

Starting Tox21 predictions with DeepChem...

Loading Tox21 data from CSV...
Dataset shape: (7831, 14)
Tasks: ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']
Featurizing training data...
Processed 1000/7831 molecules...
Processed 2000/7831 molecules...
Processed 3000/7831 molecules...
Processed 4000/7831 molecules...
Processed 5000/7831 molecules...
Processed 6000/7831 molecules...
Processed 7000/7831 molecules...
Successfully featurized 7823 training molecules
Training data shape: X=(7823, 1024), y=(7823, 12)
Training model...
  Training NR-AR...
  Training NR-AR-LBD...
  Training NR-AhR...
  Training NR-Aromatase...
  Training NR-ER...
  Training NR-ER-LBD...
  Training NR-PPAR-gamma...
  Training SR-ARE...
  Training SR-ATAD5...
  Training SR-HSE...
  Training SR-MMP...
  Training SR-p53...

Featurizing 20 query molecules...
Successfully featurized 20/20 query molecules
Making predictions...



,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53
0,0.020714,0.010514,0.136735,0.023666,0.089574,0.037979,0.017535,0.141416,0.066652,0.045342,0.089071,0.052564
1,0.019247,0.010541,0.094948,0.034347,0.094115,0.023413,0.030239,0.125769,0.050595,0.037296,0.078907,0.037930
2,0.051570,0.033400,0.116305,0.025556,0.087873,0.019591,0.011712,0.108590,0.028621,0.065245,0.117644,0.039300
3,0.041308,0.021898,0.163825,0.033119,0.110762,0.024316,0.028378,0.132531,0.095349,0.045216,0.091601,0.052118
4,0.017548,0.012191,0.186113,0.061238,0.098721,0.051221,0.015372,0.103082,0.040477,0.057636,0.124210,0.048687


In [14]:
# Add to original dataframe
EnamineAntiviralsData_wToxicity_deepchem_final = EnamineAntiviralsData_predicted.copy()
for task in tox21_tasks:
    col_name = f'DC_{task}'
    EnamineAntiviralsData_wToxicity_deepchem_final[col_name] = np.nan
    EnamineAntiviralsData_wToxicity_deepchem_final.loc[EnamineAntiviralsData_wToxicity_deepchem.index, col_name] = \
        EnamineAntiviralsData_wToxicity_deepchem[task].values

print("\n DeepChem-compatible predictions added to dataframe!")
print(f"Added columns: {[f'DC_{task}' for task in tox21_tasks]}")
EnamineAntiviralsData_wToxicity_deepchem_final


 DeepChem-compatible predictions added to dataframe!
Added columns: ['DC_NR-AR', 'DC_NR-AR-LBD', 'DC_NR-AhR', 'DC_NR-Aromatase', 'DC_NR-ER', 'DC_NR-ER-LBD', 'DC_NR-PPAR-gamma', 'DC_SR-ARE', 'DC_SR-ATAD5', 'DC_SR-HSE', 'DC_SR-MMP', 'DC_SR-p53']


,Rank,SMILES,pPotency_prediction,IC50 (M),DC_NR-AR,DC_NR-AR-LBD,DC_NR-AhR,DC_NR-Aromatase,DC_NR-ER,DC_NR-ER-LBD,DC_NR-PPAR-gamma,DC_SR-ARE,DC_SR-ATAD5,DC_SR-HSE,DC_SR-MMP,DC_SR-p53
0,1,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)N(C)CC=3C=CC=CC3,7.089,8.149146e-08,0.020714,0.010514,0.136735,0.023666,0.089574,0.037979,0.017535,0.141416,0.066652,0.045342,0.089071,0.052564
1,2,COCCN1C(=O)NC(=O)C(=C1N)N(CC=2C=CC=CC2)C(C)=O,6.840,1.445621e-07,0.019247,0.010541,0.094948,0.034347,0.094115,0.023413,0.030239,0.125769,0.050595,0.037296,0.078907,0.037930
2,3,COCCN1C=C(C=CC1=O)NC(=O)C2C(C=C(C)C)C2(C)C,6.794,1.608349e-07,0.051570,0.033400,0.116305,0.025556,0.087873,0.019591,0.011712,0.108590,0.028621,0.065245,0.117644,0.039300
3,4,COCCN1C(=NC2=C1C(=O)NC(=O)N2C)NN=CC=3C=CC=CC3,6.650,2.238847e-07,0.041308,0.021898,0.163825,0.033119,0.110762,0.024316,0.028378,0.132531,0.095349,0.045216,0.091601,0.052118
4,5,COC=1C=CC(OC)=C(C1)C(O)CN2C(C)=NC=3C=CC=CC32,6.643,2.273610e-07,0.017548,0.012191,0.186113,0.061238,0.098721,0.051221,0.015372,0.103082,0.040477,0.057636,0.124210,0.048687
5,6,CCC=1C=C(CC)N(N1)C2=NC3=C(C(=O)NC(=O)N3C)N2CCOC,6.629,2.347164e-07,0.036747,0.016545,0.162314,0.027375,0.090101,0.025017,0.023862,0.135262,0.086518,0.038822,0.106418,0.089232
6,7,CCOCCN1C(=NC2=C1C(=O)NC(=O)N2C)N3N=C(C)C=C3C,6.600,2.510905e-07,0.021897,0.022932,0.123533,0.036725,0.071555,0.025023,0.014081,0.133738,0.140694,0.036534,0.114930,0.071995
7,8,CC=1C=CC=C(N1)NC(=O)NC2CCCN(CC(F)(F)F)C2=O,6.573,2.674945e-07,0.033732,0.063781,0.101444,0.033634,0.086850,0.023653,0.055597,0.153114,0.045955,0.039446,0.095676,0.056154
8,9,CC1=CC=2N=CN(CC(O)CN3C(=O)NC(C)(C)C3=O)C2C=C1C,6.547,2.837417e-07,0.026535,0.010501,0.145244,0.029177,0.074724,0.022400,0.018525,0.105106,0.024037,0.035759,0.100122,0.080961
9,10,COC=1C=CC(=CN1)NC(=O)NCCC(=O)N2CC(C)OC(C)C2,6.529,2.959537e-07,0.037943,0.025259,0.133014,0.044183,0.079210,0.020209,0.049972,0.100834,0.021206,0.031546,0.082406,0.058672


In [19]:
# Define critical vs less critical toxicity endpoints
critical_tox = ['DC_SR-p53', 'DC_NR-AR', 'DC_NR-ER']  # DNA damage, hormone disruption
moderate_tox = ['DC_NR-AhR', 'DC_SR-ARE', 'DC_SR-MMP']
all_tox = ['DC_NR-AR', 'DC_NR-AR-LBD', 'DC_NR-AhR', 'DC_NR-Aromatase', 
           'DC_NR-ER', 'DC_NR-ER-LBD', 'DC_NR-PPAR-gamma', 'DC_SR-ARE', 
           'DC_SR-ATAD5', 'DC_SR-HSE', 'DC_SR-MMP', 'DC_SR-p53']

# Calculate weighted toxicity score
df = EnamineAntiviralsData_wToxicity_deepchem_final.copy()

# Critical endpoints (weight 2x)
df['critical_tox_score'] = df[critical_tox].mean(axis=1)

# All endpoints
df['overall_tox_score'] = df[all_tox].mean(axis=1)

# Count critical violations (> 0.7 threshold)
df['critical_violations'] = (df[critical_tox] > 0.7).sum(axis=1)

# Count any violations (> 0.5 threshold)
df['total_violations'] = (df[all_tox] > 0.5).sum(axis=1)

# Potency score (convert IC50 to µM and invert)
df['IC50_uM'] = df['IC50 (M)'] * 1e6
df['potency_score'] = np.log10(1 / df['IC50_uM'])  # Log scale for potency

# Normalize scores to 0-1
df['potency_norm'] = (df['potency_score'] - df['potency_score'].min()) / \
                     (df['potency_score'].max() - df['potency_score'].min())
df['safety_norm'] = 1 - df['overall_tox_score']

# Final composite score
df['final_score'] = (
    0.5 * df['potency_norm'] +           # 50% potency
    0.35 * df['safety_norm'] +           # 35% overall safety
    0.15 * (1 - df['critical_tox_score'])  # 15% critical endpoints
)

# Penalize molecules with critical violations
df.loc[df['critical_violations'] > 0, 'final_score'] *= 0.5

# Rank
ranked = df.sort_values('final_score', ascending=False)

print("\n=== TOP 10 DRUG CANDIDATES ===")
print(ranked[['Rank', 'SMILES', 'pPotency_prediction', 'IC50 (M)', 'overall_tox_score', 'critical_violations', 
              'total_violations', 'final_score']].head(10).to_string(index=False))



=== TOP 10 DRUG CANDIDATES ===
 Rank                                          SMILES  pPotency_prediction     IC50 (M)  overall_tox_score  critical_violations  total_violations  final_score
    1  COCCN1C(=NC2=C1C(=O)NC(=O)N2C)N(C)CC=3C=CC=CC3                7.089 8.149146e-08           0.060980                    0                 0     0.970514
    2   COCCN1C(=O)NC(=O)C(=C1N)N(CC=2C=CC=CC2)C(C)=O                6.840 1.445621e-07           0.053112                    0                 0     0.814717
    3      COCCN1C=C(C=CC1=O)NC(=O)C2C(C=C(C)C)C2(C)C                6.794 1.608349e-07           0.058784                    0                 0     0.781747
    4   COCCN1C(=NC2=C1C(=O)NC(=O)N2C)NN=CC=3C=CC=CC3                6.650 2.238847e-07           0.070035                    0                 0     0.684717
    5    COC=1C=CC(OC)=C(C1)C(O)CN2C(C)=NC=3C=CC=CC32                6.643 2.273610e-07           0.068041                    0                 0     0.683099
    6 CCC=1C=C

2.2 Use DeepChem's pre-trained Tox21 models